::: {.sb-lesson-links}
[Exercises →](../exercises/ridge-vs-lasso.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/03-regularization.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-cross-validation-and-grid-search.ipynb){download="lesson-cross-validation-and-grid-search.ipynb"}  
Data: [gene_expression_regression.csv](../downloads/data/gene_expression_regression.csv){download="gene_expression_regression.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

## What will you learn in this course? 🧐🧐

This course teaches you a model evaluation technique called cross-validation. It gives you a more precise idea of how well your model performs on unseen data. It also helps you pick the best possible hyperparameters for your model (e.g. the regularization strength).

* How to protect yourself from overfitting
* Diagnosing under/over-fitting
* Hyperparameter optimization
     * Example: tuning the regularization strength
     * Cross-validated grid search
     * Remark about computation time
         * Example 1: tuning one hyperparameter
         * Example 2: tuning several hyperparameters at the same time

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
pio.templates["course"] = go.layout.Template(layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"])
pio.templates.default = "plotly_white+course"
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_validate, GridSearchCV

## How to protect yourself from overfitting ⛔️

Underfitting does happen in practice, usually because of a lack of relevant data, and it can often be solved with a more complex model. The most common enemy of data scientists, however, is overfitting: it gives the illusion of good performance, but it is actually a trap!



::: {.callout-tip title="Intuition"}
Imagine judging a student with a single exam of 10 questions: if they are lucky with the questions, they look brilliant; if they are unlucky, they look weak. A fairer way is to give several different exams and average the grades. Cross-validation does exactly this with a model: every observation gets to be in an "exam" (the validation set) exactly once, and we look at the average grade *and* at how much the grades vary.
:::



A simple and effective way to avoid this trap is k-fold cross-validation (other cross-validation techniques exist, but we will use this one throughout this course). You choose an integer k (often 5 or 10; the default depends on the library), and randomly split the observations into k groups of equal size. Then the following steps are repeated k times:

* Set aside one group i among the k groups: this is the **validation set**. The k-1 other groups together form the **training set**.
* Fit the chosen model on the **training set**.
* Compute the error of this model i on the **validation set** (group i).

Comparing the training error with the validation error shows the real predictive power of a model: it compares the model's performance on several sets of unseen data with its performance on the data it has already seen.

A single train/test split only tests the model on one particular split of the data. With cross-validation, we measure the model's performance on several random splits and see how it performs on average. Cross-validation tells you how well the model performs, and also how stable its performance is when the data changes.

![overfitting](../img/cross_validation.svg)


The figure above illustrates k-fold cross-validation. Each iteration gives a training error and a validation error, which are used to evaluate the model. These errors are usually computed with the cost function used to fit the model, or simply with the mean squared error.

### See it with data 🔬: what k-fold actually does

Let's take a tiny "dataset" of 10 observations, numbered 0 to 9, and ask scikit-learn's `KFold` to split it into k = 5 folds. For each fold, we print which observations are used for training and which ones are kept aside for validation.

In [ ]:
tiny = np.arange(10)                       # 10 observations: 0, 1, ..., 9
kf = KFold(n_splits=5, shuffle=True, random_state=0)

for i, (train_idx, val_idx) in enumerate(kf.split(tiny), start=1):
    print(f"Fold {i}: train on {train_idx}  |  validate on {val_idx}")

Fold 1: train on [0 1 3 4 5 6 7 9]  |  validate on [2 8]
Fold 2: train on [0 1 2 3 5 6 7 8]  |  validate on [4 9]
Fold 3: train on [0 2 3 4 5 7 8 9]  |  validate on [1 6]
Fold 4: train on [0 1 2 4 5 6 8 9]  |  validate on [3 7]
Fold 5: train on [1 2 3 4 6 7 8 9]  |  validate on [0 5]


**What we see:** each fold trains on 8 observations and validates on the 2 others. Look at the "validate on" column: every observation from 0 to 9 appears there exactly once. So, at the end, each observation has been used once to *test* the model and 4 times to *train* it.

### See it with data 🔬: one split is a lottery, k-fold is a survey

Now a real dataset: the expression of 2,000 genes for 801 patients, and a continuous `target` to predict (the dataset of the previous lesson). We evaluate the same Ridge model with 20 different random train/test splits (only the `random_state` changes), then with a single 5-fold cross-validation.

In [ ]:
genes = pd.read_csv("../data/gene_expression_regression.csv")
X, y = genes.drop(columns="target"), genes["target"]
model = make_pipeline(StandardScaler(), Ridge(alpha=1000))

split_scores = []
for seed in range(20):                     # 20 different random splits
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    split_scores.append(model.fit(X_tr, y_tr).score(X_te, y_te))   # R² on the test part
split_scores = pd.Series(split_scores, name="R² test")
print(f"20 single splits: R² from {split_scores.min():.3f} to {split_scores.max():.3f}")

cv_scores = cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=0))
print("5-fold CV scores:", cv_scores.round(3))
print(f"5-fold CV: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

20 single splits: R² from 0.563 to 0.729


5-fold CV scores: [0.611 0.706 0.735 0.615 0.669]
5-fold CV: 0.667 ± 0.049


In [ ]:
fig = px.strip(split_scores, orientation="h", title="Same model, 20 random train/test splits",
               labels={"value": "R² on the test set", "variable": ""})
fig.add_vline(x=cv_scores.mean(), line_dash="dash", annotation_text="5-fold CV mean")
fig.show()

**What we see:** with exactly the same model and data, a single split gives an R² anywhere between 0.563 and 0.729, depending only on the luck of the split! If you had tried only one seed, you could have concluded that the model is mediocre or quite good. Cross-validation gives one more reliable answer, 0.667 ± 0.049: the average over 5 validation sets, and an idea of how much the score varies from one set to another.

## Diagnosing under/over-fitting 🩺🩺
To detect underfitting or overfitting, compare the train and test (or train and validation) scores:

* If $score(train) \sim score(test) \sim 0$: the model is underfitting
* If $score(train) >> score(test)$: the model is overfitting
* If $score(train) \sim score(test) >> 0$: the model is just right!

> 👋 $>>$ means *much greater than*

### See it with data 🔬: three diagnoses

`cross_validate` with `return_train_score=True` gives both the training score and the validation score of each fold. Let's compare three models on the gene dataset: a Ridge model with an enormous penalty, a plain linear regression (2,000 variables for only 640 training patients per fold), and a Lasso model.

In [ ]:
candidates = {
    "Ridge, alpha=1,000,000": make_pipeline(StandardScaler(), Ridge(alpha=1e6)),
    "Linear regression": LinearRegression(),
    "Lasso, alpha=0.3": make_pipeline(StandardScaler(), Lasso(alpha=0.3, max_iter=10000)),
}
rows = []
for name, candidate in candidates.items():
    res = cross_validate(candidate, X, y, cv=5, return_train_score=True)
    rows.append({"model": name, "R² train": res["train_score"].mean(),
                 "R² validation": res["test_score"].mean()})
pd.DataFrame(rows).round(3)

,model,R² train,R² validation
0,"Ridge, alpha=1,000,000",0.061,0.047
1,Linear regression,1.000,0.331
2,"Lasso, alpha=0.3",0.915,0.865


**What we see:** the three rules above, one per line.

* The very penalized Ridge has R² train ≈ 0.061 and R² validation ≈ 0.047: both close to 0, it is **underfitting** (all its coefficients are crushed to almost 0).
* The linear regression has a perfect R² train of 1.0 but only 0.331 in validation: score(train) >> score(validation), it is **overfitting**.
* The Lasso model has 0.915 in training and 0.865 in validation: both high and close to each other, it is **just right**.

## Hyperparameter optimization 🔧🔧

In machine learning, hyperparameter optimization (or tuning) is the problem of choosing the best set of hyperparameters for a learning algorithm.



::: {.callout-tip title="Intuition"}
Think of a student preparing for an exam. *How many hours per day* they study and *which textbook* they use are decided before studying starts: these are hyperparameters. *What they actually learn* comes out of the studying: these are the parameters. Grid search is like trying several study plans, and comparing them with practice exams (cross-validation) before the real exam.
:::



A hyperparameter is a parameter that controls the learning process, but is not directly used to compute the model's predictions. Hyperparameters are not learned during training: they have to be set beforehand. By contrast, the other parameters (typically, the coefficients of the model) are learned during training.

### Example: tuning the regularization strength
An example of a model hyperparameter is the regularization strength (noted either $\lambda$ or $\alpha$, depending on the convention) in regularized linear regression models. The value of $\alpha$ has to be tuned to find the right model complexity, and get a performance that is "just right" between underfitting and overfitting.

### Cross-validated grid search
The traditional way to optimize hyperparameters is grid search: an exhaustive search through a manually chosen set of hyperparameter values. A grid search must be guided by a performance metric, typically measured by cross-validation on the training set.

The cross-validated grid search algorithm follows these steps:

* Choose the list of hyperparameter values to test (for example: $\alpha = [0.1, 0.5, 1.0]$)
* For each combination of hyperparameters, run k-fold CV to estimate the generalization score of this combination
* Pick the combination of hyperparameters with the highest cross-validated score
* Re-train the model with this combination, on the whole training set

![grid_search](../img/grid_search.png)

### See it with data 🔬: a grid search on two hyperparameters

Let's tune two hyperparameters at the same time: the degree of a polynomial regression (model complexity) and the Ridge penalty `alpha`. The data are 300 noisy points around the curve $y = \sin(3x)$. We keep 20% of the points as a test set, and run a 5-fold grid search on the training set only.

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(-1, 1, 300)
y_curve = np.sin(3 * x) + rng.normal(0, 0.3, 300)
x_train, x_test, y_train, y_test = train_test_split(x.reshape(-1, 1), y_curve, test_size=0.2, random_state=0)

pipe = make_pipeline(PolynomialFeatures(), StandardScaler(), Ridge())
param_grid = {"polynomialfeatures__degree": range(1, 9),
              "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(pipe, param_grid, cv=5).fit(x_train, y_train)
print("Best hyperparameters:", grid.best_params_)
print(f"Best cross-validated R²: {grid.best_score_:.3f}")

Best hyperparameters: {'polynomialfeatures__degree': 6, 'ridge__alpha': 0.1}
Best cross-validated R²: 0.848


In [ ]:
scores = pd.DataFrame(grid.cv_results_).pivot(index="param_ridge__alpha",
                                              columns="param_polynomialfeatures__degree",
                                              values="mean_test_score")
fig = px.imshow(scores, text_auto=".2f", aspect="auto", color_continuous_scale="Blues",
                title="Cross-validated R² for each combination of hyperparameters",
                labels={"x": "polynomial degree", "y": "alpha", "color": "mean R²"})
fig.update_yaxes(type="category")
fig.show()

**What we see:** each cell of the heatmap is one combination, scored by 5-fold cross-validation. Degrees 1 and 2 stay at about 0.56 whatever the penalty (underfitting: a straight line or a parabola cannot follow a sine wave), and `alpha = 100` holds every model back (R² between 0.48 and 0.53). The best combination, a degree-6 polynomial with `alpha = 0.1`, reaches a cross-validated R² of 0.848, but every cell with a degree of at least 3 and `alpha` ≤ 1 scores between 0.838 and 0.848. This is typical: there is often a "plateau" of good hyperparameters rather than a single magic value.

### See it with data 🔬: why the test set must stay untouched

The grid search above never saw the test set. So we can use it, *once*, to check that the cross-validated score was honest:

In [ ]:
print(f"Cross-validated R² (training set): {grid.best_score_:.3f}")
print(f"R² on the untouched test set:      {grid.score(x_test, y_test):.3f}")

Cross-validated R² (training set): 0.848
R² on the untouched test set:      0.798


The two scores are close (0.848 and 0.798): the cross-validated score was a fair estimate of the performance on new data. The small gap is normal, since the test set only has 60 points and carries its own bit of luck. Now, what if the test set had influenced our choices? Here is an extreme example: the target and the 1,000 explanatory variables are **pure random noise**, so no model can truly predict anything. We select the 10 variables most correlated with the target, then fit a linear regression on the training set. The only difference between the two lines is which rows were used to *select* the variables.

In [ ]:
r = np.random.default_rng(2)
X_noise, y_noise = r.normal(size=(100, 1000)), r.normal(size=100)   # pure noise!
train_rows, test_rows = train_test_split(np.arange(100), test_size=0.3, random_state=0)

def top_10_variables(rows):
    corr = [abs(np.corrcoef(X_noise[rows, j], y_noise[rows])[0, 1]) for j in range(1000)]
    return np.argsort(corr)[-10:]

for label, rows in [("selected on ALL rows (test included)", np.arange(100)),
                    ("selected on training rows only", train_rows)]:
    cols = top_10_variables(rows)
    reg = LinearRegression().fit(X_noise[train_rows][:, cols], y_noise[train_rows])
    print(f"{label:38s} -> R² test = {reg.score(X_noise[test_rows][:, cols], y_noise[test_rows]):.3f}")

selected on ALL rows (test included)   -> R² test = 0.428


selected on training rows only         -> R² test = -0.052


**What we see:** when the test rows helped choose the variables, the model gets an R² of 0.428 on the test set... on pure noise! The test set "leaked" into the choices, so it is no longer a fair exam. When the choice is made on the training rows only, the test R² is -0.052, the honest answer: there is nothing to predict. The same thing happens, more quietly, if you try many hyperparameters and keep the one with the best *test* score. Choose with cross-validation on the training set, and look at the test set only at the very end.

### Remark about computation time
Cross-validated grid search can be very time-consuming, because the model is trained many times! For k-fold cross-validation, the number of model trainings $n$ is:

$$
n = k \times n_{comb}
$$

where $n_{comb}$ is the total number of combinations of hyperparameters values to be tested.

#### Example 1: tuning one hyperparameter
Let's imagine we want to tune the regularization strength $\alpha$ and test 3 different values: $\alpha = [0.1, 0.5, 1.0]$. We want to use 5-fold cross-validation.

Then, the total number of trainings in the cross-validated grid search is:

$$
n = 5 \times 3 = 15
$$

### See it with data 🔬: counting the trainings

With `verbose=1`, `GridSearchCV` tells you how many models it is going to train. Let's run Example 1 (3 values of $\alpha$, 5 folds) on our curve data:

In [ ]:
small_grid = GridSearchCV(make_pipeline(PolynomialFeatures(5), StandardScaler(), Ridge()),
                          {"ridge__alpha": [0.1, 0.5, 1.0]}, cv=5, verbose=1)
small_grid.fit(x_train, y_train);

Fitting 5 folds for each of 3 candidates, totalling 15 fits


**What we see:** scikit-learn announces "5 folds for each of 3 candidates, totalling 15 fits", exactly $n = 5 \times 3 = 15$. (It then trains one last model with the best $\alpha$ on the whole training set, as explained in the last step of the algorithm.)

#### Example 2: tuning several hyperparameters at the same time
It is very common to tune several hyperparameters at the same time. Let's imagine we want to tune three hyperparameters (respectively named $\alpha$, $\beta$ and $\gamma$) and we want to try the following values:

* $\alpha = [0.1, 0.5, 1.0]$ 
* $\beta = [1, 2, 3, 4]$
* $\gamma = [-1, 1]$

The numbers of values to test for each hyperparameter are:

* $n_\alpha = 3$
* $n_\beta = 4$
* $n_\gamma = 2$

Then, the total number of combinations to test is:

$$
n_{comb} = 3 \times 4 \times 2 = 24
$$

If we want to perform 5-fold cross-validation, the total number of trainings in the cross-validated grid search is:

$$
n = 5 \times 24 = 120
$$

## Resources 📚📚

* [Cross Validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html)

* [Kfold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html#sklearn.model_selection.KFold)

* [Exhaustive Grid Search](https://scikit-learn.org/stable/modules/grid_search.html#exhaustive-grid-search)

::: {.sb-next}
**[Next: Regularization with scikit-learn →](../lessons/workflow.ipynb)**
:::